In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc 
gc.collect()

47

In [3]:
import pandas as pd
import numpy as np
import getpass
import sys
import datetime
import io
usr_name = getpass.getuser()
sys.path.append(f'/home/{usr_name}/notebooks/utils')
from spark_utils import *
#import datadicts as dd
from bpm_features import calc_all_features

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [4]:
import os
import sys
import numpy as np
import pandas as pd
# import polars as pl
  
def get_spark_session(name, level):
    """
    Get spark context
    :: name - set your app name
    :: level - set max resources level
    """
    python_path = sys.executable
    kernel = python_path.split('/')[-3]
    os.environ['SPARK_MAJOR_VERSION'] = '3'
    os.environ['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
    os.environ['PYSPARK_DRIVER_PYTHON'] = python_path
    os.environ['PYSPARK_PYTHON'] = python_path
    os.environ['LD_LIBRARY_PATH'] = '/opt/python/virtualenv/jupyter/lib'
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/')
    sys.path.insert(0, '/usr/sdp/current/spark3-client/python/lib/py4j_current')
 
    # Resources Level Profiles                           #  cpu --  ram -- desc
    if level == 1: lv = ['basic',2,10,2,10,2,2,10]       #   21 --  142 -- для базовых запросов (show create table tbl, show partitions tbl)
    if level == 2: lv = ['basic+CPU',2,10,2,10,2,2,20]   #   41 --  262 -- для простой аналитики (select * from limit 100, sum/count/avg)
    if level == 3: lv = ['middle',4,28,6,28,6,4,20]      #   81 --  742 -- для агрегатов за период 1-2мес (client_aggr_mnth, epk_campaign_daily)
    if level == 4: lv = ['middle+CPU',4,28,6,28,6,4,25]  #  101 --  912 -- для агрегатов за период >1-6мес  (client_aggr_mnth, epk_campaign_daily)
    if level == 5: lv = ['high',4,28,6,36,8,6,30]        #  121 -- 1100 -- для детальных таблиц с большими партициями (_sbol, _card, _eps)
    if level == 6: lv = ['high+CPU',4,18,5,36,8,6,40]    #  161 -- 1000 -- для детальных таблиц с мелкими партициями (feedbacks)
    if level == 7: lv = ['unfriendly',5,28,6,44,10,8,40] #  201 -- 1458 -- для запуска вечером/ночью или на пустом кластере (не рекомендуется)
    lvname = f'{level}.{lv[0]}({lv[1]*lv[7]+1},{lv[4]+lv[2]*lv[7]})'
    print(f'Kernel: {kernel}, Python_path: {python_path}, Resource_level: {lvname}')
    
    # Spark Config      
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
  
    conf = SparkConf().setAppName(f'{name} \n ::{kernel}::{lvname}::')\
        .setMaster("yarn")\
        .set('spark.executor.cores',                     f'{lv[1]}')\
        .set('spark.executor.memory',                    f'{lv[2]}g')\
        .set('spark.executor.memoryOverhead',            f'{lv[3]}g')\
        .set('spark.driver.memory',                      f'{lv[4]}g')\
        .set('spark.driver.memoryOverhead',              f'{lv[5]}g')\
        .set('spark.driver.maxResultSize', '10g')\
        .set('spark.dynamicAllocation.initialExecutors', f'{lv[6]}')\
        .set('spark.dynamicAllocation.maxExecutors',     f'{lv[7]}')\
        .set('spark.dynamicAllocation.enabled', 'true')\
        .set('spark.dynamicAllocation.executorIdleTimeout', '120s')\
        .set('spark.dynamicAllocation.cachedExecutorIdleTimeout', '600s')\
        .set('spark.hive.mapred.supports.subdirectories', 'true')\
        .set('spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive', 'true')\
        .set('spark.shuffle.service.enabled', 'true')\
        .set('spark.port.maxRetries', '150')\
       .set('spark.sql.parquet.writeLegacyFormat', 'true')\
        .set('spark.kerberos.access.hadoopFileSystems','hdfs://arnsdpsbx:8020/')\
        .set('spark.sql.autoBroadcastJoinThreshold','20971520')
    
    spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
    return spark

try: spark
except NameError: print('Spark3 Starting')
else:
    print('Spark3 Restarting')
    spark.stop()
    
spark = get_spark_session('platon_features_agent', 5) # For example, MyPySpark3
  
import pyspark.sql.functions as sf
from pyspark.sql.types import *
  
sc = spark.sparkContext
sc.setLogLevel('OFF')  # or 'INFO' or 'WARN' or 'OFF'
spark


Spark3 Starting
Kernel: mlpy3811v23, Python_path: /data/sdp/mlpy3811v23/bin/python, Resource_level: 5.high(121,876)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 08:39:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/11 08:39:18 WARN HiveConf: HiveConf of name hive.mapred.supports.subdirectories does not exist
26/03/11 08:39:19 WARN Client: Exception encountered while connecting to the server 
org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
	at org.apache.hadoop.security.SaslRpcClient.saslConnect(SaslRpcClient.java:376)
	at org.apache.hadoop.ipc.Client$Connection.setupSaslConnection(Client.java:623)
	at org.apache.hadoop.ipc.Client$Connection.access$2300(Client.java:414)
	at org.apache.hadoop.ipc.Client$Connection$2.run(Client.java:832)
	at org.apache.hadoop.ipc.Client$Connection$2.run(Client.java:828)
	at java.security.AccessController.doPri

In [5]:
df = pd.read_parquet('target_new.parquet')

In [6]:
df = df.fillna(0)

In [7]:
spark_df = spark.createDataFrame(df)
spark_df.createOrReplaceTempView('target_df')

/usr/sdp/current/spark3-client/python/pyspark/sql/pandas/conversion.py:371: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():


In [8]:
target = spark.sql('''
select epk_id, last_day(date(start_dt) - interval 1 month) as report_dt, unique_only_nflag as target
from target_df
''')

target.createOrReplaceTempView('target')
target.count()

91854

In [9]:
tables_dict = {'agg':'prx_bpm_client_aggr_custom_rozn_client_aggr.ft_client_aggr_mnth'}
               #'feedbacks':'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_feedbacks',
               #'vsp_visits':'prx_bpm_visiting_vsp_custom_rozn_sscxdata_cxdm.cxdm_visiting_vsp_v2',
               #'card_transactions': 'prx_bpm_feedbacks_prom_custom_rozn_cx_ml_features.mon_card_transactions',
               #'e_cod':'prx_bpm_cod_platform_cod.cod_deposit_deposit',
               #'idoc': 'prx_bpm_arrests_internal_aiv_deposit.idoc',
               #'idoc_acc': 'prx_bpm_arrests_internal_aiv_deposit.idoc_acc',
               #'pos_embeddings_fl': 'prx_bpm_pos_emb_custom_rozn_ml360.u_fl_transaction_embeddings' ,
               #'embeddings_fl': 'prx_bpm_multimodel_emb_custom_fin_palm_ml.cmn_multimodal_emb_ind_fct'}


In [10]:
model_name = 'bpm_agent_py_model_paid_new'
output_scheme = 'arnsdpsbx_team_ss' 
mode = 'append'

In [14]:
calc_all_features(spark, target, tables_dict, model_name, output_scheme, mode)

agg_features: done
flows12_features: done
pos_dynamic_features: done


aggr_dynamic_features_1: done


aggr_dynamic_features_2: done


aggr_dynamic_features_3: done


all_features_train: done
arnsdpsbx_team_ss.bpm_agent_py_model_paid_new_all_features_train


100%|██████████| 133/133 [10:09<00:00,  4.58s/it]


all_features_fix: done
arnsdpsbx_team_ss.bpm_agent_py_model_paid_new_all_features_fix_types_train: done


In [5]:
spark.sql('''
select * from arnsdpsbx_team_ss.bpm_agent_py_model_paid_new_all_features_fix_types_train
''').write.parquet('hdfs://arnsdpsbx/user/team/team_ss/bpm_agent_py_model_paid_new_all_features_fix_types_train_model', mode='overwrite')